In [66]:
# AutoEIT GSoC 2026 - Evaluation Test II
# Automated Scoring System for Elicited Imitation Task

"""
This notebook implements a reproducible scoring system that compares
learner responses with target sentences and assigns scores based on similarity.

Approach:
- Text normalization (lowercase, remove punctuation)
- Similarity calculation using SequenceMatcher
- Rule-based scoring system
"""

'\nThis notebook implements a reproducible scoring system that compares\nlearner responses with target sentences and assigns scores based on similarity.\n\nApproach:\n- Text normalization (lowercase, remove punctuation)\n- Similarity calculation using SequenceMatcher\n- Rule-based scoring system\n'

In [67]:
import pandas as pd
from difflib import SequenceMatcher
import re

In [68]:
def clean(text):
    text = str(text).lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

In [69]:
def similarity(a, b):
    return SequenceMatcher(None, clean(a), clean(b)).ratio()

def score_sentence(target, learner):
    t = clean(target)
    l = clean(learner)

    # sequence similarity (captures order)
    sim = SequenceMatcher(None, t, l).ratio()

    target_words = t.split()
    learner_words = l.split()

    missing = len(set(target_words) - set(learner_words))
    extra = len(set(learner_words) - set(target_words))

    penalty = 0.05 * (missing + extra)

    final = sim - penalty

    if final > 0.9:
        return 5
    elif final > 0.75:
        return 4
    elif final > 0.6:
        return 3
    elif final > 0.4:
        return 2
    else:
        return 1

In [70]:
file = "../data/AutoEIT Sample Transcriptions for Scoring.xlsx"
sheets = pd.read_excel(file, sheet_name=None)

print("Loaded sheets:", list(sheets.keys()))

Loaded sheets: ['Info', '38001-1A', '38002-2A', '38004-2A', '38006-2A']


In [71]:
output_sheets = {}

for sheet_name, df in sheets.items():
    
    df.columns = df.columns.str.strip()

    # skip sheets without required columns
    if "Stimulus" not in df.columns or "Transcription Rater 1" not in df.columns:
        continue

    df["Score"] = df.apply(
        lambda x: score_sentence(x["Stimulus"], x["Transcription Rater 1"]),
        axis=1
    )
    
    output_sheets[sheet_name] = df

print("Scoring complete!")

for name, df in output_sheets.items():
    print(f"{name} score distribution:")
    print(df["Score"].value_counts())
    print("------")

Scoring complete!
38001-1A score distribution:
Score
3    8
2    7
4    6
5    5
1    4
Name: count, dtype: int64
------
38002-2A score distribution:
Score
1    19
2     5
3     3
5     2
4     1
Name: count, dtype: int64
------
38004-2A score distribution:
Score
1    10
2     9
5     4
4     4
3     3
Name: count, dtype: int64
------
38006-2A score distribution:
Score
1    21
2     5
3     2
4     2
Name: count, dtype: int64
------


In [72]:
output_file = "../output/output_scores.xlsx"

with pd.ExcelWriter(output_file) as writer:
    for name, df in output_sheets.items():
        df.to_excel(writer, sheet_name=name, index=False)

print("Saved to:", output_file)

Saved to: ../output/output_scores.xlsx


In [73]:
# Show one sheet preview
list(output_sheets.values())[0].head()

,Sentence,Stimulus,Transcription Rater 1,Score
0,1,Quiero cortarme el pelo (7),Quiero cortarme el pelo,5
1,2,El libro está en la mesa (7),El libro está en la mesa,5
2,3,El carro lo tiene Pedro (8),El carro lo tiene Pedro,5
3,4,El se ducha cada mañana (9),El se ducha cada mañana,5
4,5,¿Qué dice usted que va a hacer hoy? (9),Que dices ustedes se que van a hacer hoy?,2


### Observations

The scoring system shows clear variability across participants, with some participants receiving predominantly low scores while others show a balanced distribution.

This indicates that the model is able to differentiate between varying levels of transcription quality and linguistic accuracy, aligning with the goal of consistent and meaningful evaluation.